# Chat with a code or literature project

Choose a project, ask questions, inspect the exact evidence, and continue or
resume the same conversation.

## 1. Prepare QAChat

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

## 2. Choose the project

Normally, you only need to edit these two settings:

- **`PROJECT_KIND`** chooses **code** or **literature**.
- **`PROJECT_DIRECTORY`** chooses the source or QAChat project folder.
- Leave the directory as **None** to use the matching sandbox example.


In [ ]:
PROJECT_KIND = "code"  # "code" or "litarature"
PROJECT_DIRECTORY = None  # "/path/to/my-qachat-project"


In [ ]:
if PROJECT_KIND not in {"code", "literature"}:
    raise ValueError("PROJECT_KIND must be `code` or `literature`.")


In [ ]:
if PROJECT_DIRECTORY is None:
    project_directory = root / "sandbox" / f"{PROJECT_KIND}_qa"
else:
    project_directory = Path(PROJECT_DIRECTORY).expanduser()

print(f"Project: {project_directory}")


If the selected folder has no **profile.ini**, QAChat creates one with
default settings. It uses a **repository/** or **documents/** subfolder
when present; otherwise it reads the selected folder itself.


## 3. Open or resume QAChat

The first open creates any missing project settings, reads the source,
builds reusable indexes, and starts a new conversation. Later opens reuse
current indexes and restore the active conversation.

In [ ]:
profile_path = project_directory / "profile.ini"


In [ ]:
from qachat import initialize_project


project = initialize_project(
    project_directory,
    PROJECT_KIND,
)

if profile_path.is_file():
    print(f"Found project settings: {profile_path}")
else:
    print(f"Created project settings: {profile_path}")


In [ ]:
index_path = project.indexes / "retrieval.json"


In [ ]:
from qachat import RAGSession


if "session" in globals():
    session.close()

session = RAGSession(project.root)


In [ ]:
if session.turns:
    print("Resumed a previous conversation.")
    print(f"Restored turns: {len(session.turns)}")
    print(f"Last question: {session.turns[-1].question}")
elif index_path.is_file():
    print("Opened an existing project and started a new conversation.")
else:
    print("Opened this project for the first time.")
    print("Built its indexes and started a new conversation.")

print(f"Index status: {session.index_status}")
print(f"Backend: {session.backend}")
print(f"Source: {session.project.source}")

## 4. Ask a question

Edit **QUESTION**, then run the following answer cell. QAChat immediately
saves the question, answer, evidence, and settings.

In [ ]:
DEFAULT_QUESTIONS = {
    "code": "How does this project convert Celsius to Kelvin?",
    "literature": "What did the toy sleep study observe?",
}

QUESTION = DEFAULT_QUESTIONS[session.project.profile.kind]

print(f"Question: {QUESTION}")



In [ ]:
answer = session.ask(QUESTION)

print(f"Answer: {answer}")


## 5. Inspect the evidence

The latest turn contains the exact source chunks given to the LLM. Their
numbers match the citations requested in the answer.

In [ ]:
latest_turn = session.turns[-1]

for item in latest_turn.evidence:
    location = f"{item.path}:{item.start_line}-{item.end_line}"
    print(f"[{item.citation}] {location}")
    print(item.text)
    print()


## 6. Keep asking questions

Edit **NEXT_QUESTION** and run the next cell with Shift+Enter. Repeat that
same cell for every new question. Keep the session open so each answer can
use the earlier dialogue as conversation history.

In [ ]:
NEXT_QUESTION = "Can you explain that in more detail?"

print(f"Next question: {NEXT_QUESTION}")

In [ ]:
next_answer = session.ask(NEXT_QUESTION)
latest_turn = session.turns[-1]

print(f"Answer: {next_answer}")


In [ ]:
print("Evidence:")
for item in latest_turn.evidence:
    location = f"{item.path}:{item.start_line}-{item.end_line}"
    print(f"[{item.citation}] {location}")


## 7. Find the saved conversation and settings

**profile.ini** contains the reusable project settings.


In [ ]:
profile_path = session.project.root / "profile.ini"

print(f"Project settings: {profile_path}")


**conversation.json** contains every question, answer, evidence set, and
parameter snapshot in the active conversation.


In [ ]:
conversation_path = session.project.logs / "conversation.json"

print(f"Active conversation: {conversation_path}")


In [ ]:
print(f"Reusable indexes: {session.project.indexes}")


In [ ]:
import json

print(json.dumps(latest_turn.parameters, indent=2, sort_keys=True))

## 8. Optionally start a new conversation

Set this option to **True** and run the next cell once. QAChat renames the
active log with a UTC timestamp, clears the conversation from memory, and
keeps the reusable indexes.

In [ ]:
START_NEW_CONVERSATION = True

if START_NEW_CONVERSATION:
    archived_path = session.reset()
    print(f"Archived conversation: {archived_path}")
    print(f"Active turns: {len(session.turns)}")

## 9. Close QAChat

Closing releases the models without removing indexes or conversations.

In [ ]:
session.close()

print("QAChat closed safely.")

## Result

The selected project is ready for continued questions. Opening it again
restores the active conversation and displays the last question.